# H_TextMask — 패턴 단어 마스킹

Purchase-level Recall on the fixed changed pattern products. The preparation cell is reconstructed from the conversation and frozen JSON. Embedding/training/evaluation sources survived. Default mode reads saved runs.

**탐색적 분석.** Uploaded notebook cells were reorganized after the experiment. Saved CSV/JSON/bootstrap arrays are unchanged. Outputs and execution counts have been cleared. The core is pinned to the recorded experiment commit; dependency versions may still differ.


In [ ]:
HYPOTHESIS = "H_TextMask"
# Fresh Colab runtime: run this setup once per notebook.
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/msrs_paper (1)"  # Change for your shortcut.
drive.mount("/content/drive")

CORE_COMMIT = "e6b73dc4a8aee549b4310cf4f3266e373b8632ca"
CORE_DIR = Path("/content/msrs_kwangsuk_core")
REPO_URL = "https://github.com/3sunghynix/multimodal-recsys.git"
if any(name.startswith("msrs_paper") for name in sys.modules):
    raise RuntimeError("Restart the Colab session before running setup again.")
if not CORE_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CORE_DIR)], check=True)
    subprocess.run(["git", "-C", str(CORE_DIR), "checkout", "--detach", CORE_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ["git", "-C", str(CORE_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != CORE_COMMIT:
    raise RuntimeError("The existing core checkout has a different commit; use a fresh runtime.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(CORE_DIR / "paper/requirements.txt")
], check=True)
sys.path.insert(0, str(CORE_DIR / "paper"))
from msrs_paper.config import load_all, Paths
from msrs_paper import metrics
from msrs_paper.utils import verify_manifest
cfg, exps, plan = load_all()
paths = Paths(DRIVE_ROOT)  # Do not create a missing shared-data folder.
assert (paths.root / "manifest.json").is_file(), "Check DRIVE_ROOT and the shared shortcut."
manifest = verify_manifest(paths)
print("Verified files:", len(manifest["files"]))
print("Core commit:", actual_commit)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_OUTPUT_ROOT = Path("/content/drive/MyDrive/kwangsuk_reproductions") / stamp / HYPOTHESIS

def historical_folder(suffix, timestamp):
    candidates = [
        p / timestamp for p in Path("/content/drive/MyDrive").iterdir()
        if p.is_dir() and p.name.endswith(suffix) and (p / timestamp).is_dir()
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"Set the historical input folder explicitly: {candidates}")
    return candidates[0]


In [ ]:
# Default: re-score saved runs; no embedding extraction or training.
RUN_TRAINING = False
if RUN_TRAINING:
    EXP_ROOT = RUN_OUTPUT_ROOT / "experiment"
else:
    EXP_ROOT = historical_folder("text_mask_experiment", "20261014T073142381530Z")
source_paths = Paths(DRIVE_ROOT)
print("Mask experiment inputs:", EXP_ROOT)


## 1. 입력 준비
Runs only when `RUN_TRAINING=True`; creates a fresh experiment folder. Historical parquet bytes cannot be assumed identical across library versions.

In [ ]:
if RUN_TRAINING:
    # RECONSTRUCTED preparation cell; original cell was missing.
    # Frozen regex recovered from the archived experiment_plan.json.
    import re
    MASK_REGEX = '\\b(stripe[sd]?|print(ed|s)?|check(ed)?|pattern(ed)?|dot(s|ted)?|polka|floral|flower(s|ed)?|logo|graphic|motif|plaid|tartan|leopard|animal|camouflage|paisley|tie-dye|slogan|embroider(ed|y))\\b'
    pattern_rule = re.compile(MASK_REGEX, flags=re.IGNORECASE)
    mask_items = pd.read_parquet(paths.processed / "items.parquet")
    mask_items = mask_items.sort_values("idx").reset_index(drop=True)
    assert np.array_equal(mask_items["idx"].to_numpy(), np.arange(len(mask_items)))
    mask_items["original_text"] = mask_items["text"].fillna("").astype(str)
    mask_items["modified_text"] = mask_items["original_text"].map(
        lambda text: pattern_rule.sub("[MASK]", text)
    )
    mask_items["text_changed"] = mask_items["original_text"].map(
        lambda text: bool(pattern_rule.search(text))
    )
    primary_items = mask_items.loc[
        mask_items["appearance"].eq("pattern") & mask_items["text_changed"]
    ].copy()
    assert len(primary_items) == 5972, "The input catalog or frozen keyword rule differs."
    assert not mask_items["modified_text"].str.strip().eq("").any()
    EXP_ROOT.mkdir(parents=True, exist_ok=False)
    mask_items[["idx", "article_id", "appearance", "original_text", "modified_text", "text_changed"]].to_parquet(
        EXP_ROOT / "text_versions.parquet", index=False
    )
    primary_items[["idx", "article_id"]].to_csv(EXP_ROOT / "evaluation_products.csv", index=False)
    experiment_plan = {'hypothesis_id': 'H_TextMask', 'status': 'exploratory; main test data previously examined', 'hypothesis': 'Masking pattern-related words increases image contribution.', 'text_change': 'Replace each regex match with [MASK]', 'mask_regex': '\\b(stripe[sd]?|print(ed|s)?|check(ed)?|pattern(ed)?|dot(s|ted)?|polka|floral|flower(s|ed)?|logo|graphic|motif|plaid|tartan|leopard|animal|camouflage|paisley|tie-dye|slogan|embroider(ed|y))\\b', 'evaluation': {'split': 'main', 'segment': 'non_repurchase', 'appearance': 'pattern', 'products': 'Fixed products whose original text matched the regex', 'n_products': 5972, 'k': 300, 'measure': 'Purchase-level Recall@300', 'contrast': '(masked real - masked type-shuffled) - (original real - original type-shuffled)'}, 'training_seeds': [42, 43, 44], 'bootstrap': {'unit': 'customer', 'samples': 1000, 'seed': 42, 'ci': 0.95}, 'preprocessing': 'Fit one PCA on original text embeddings; apply it to both original and masked embeddings; L2-normalize both.', 'training': 'Retrain all four conditions with the same settings', 'code_commit': 'e6b73dc4a8aee549b4310cf4f3266e373b8632ca', 'created_at': '20261014T073142381530Z'}
    experiment_plan["created_at"] = datetime.now(timezone.utc).isoformat()
    experiment_plan["reconstruction_note"] = (
        "New reproduction inputs created by a reconstructed preparation cell; not a new preregistration."
    )
    with open(EXP_ROOT / "experiment_plan.json", "w", encoding="utf-8") as f:
        json.dump(experiment_plan, f, ensure_ascii=False, indent=2)
    print("Prepared evaluation products:", len(primary_items))
    print("New experiment directory:", EXP_ROOT)
else:
    print('Using historical text inputs; preparation skipped.')


## 2. 임베딩
Shared PCA fitted on original text; L2 normalization for both versions.

In [ ]:
if RUN_TRAINING:
    import numpy as np
    import json
    import shutil
    import joblib
    import torch

    from pathlib import Path
    from sklearn.decomposition import PCA
    from msrs_paper import embeddings

    # Reload the saved experiment inputs so preparation is reproducible.
    # EXP_ROOT is configured above.

    text_versions = pd.read_parquet(EXP_ROOT / "text_versions.parquet")
    text_versions = text_versions.sort_values("idx").reset_index(drop=True)

    # Embedding row numbers must match the product indices.
    assert np.array_equal(
        text_versions["idx"].to_numpy(),
        np.arange(len(text_versions))
    ), "Product index ordering is unexpected."

    source_paths = Paths(DRIVE_ROOT)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model_name = cfg["embedding"]["text_model"]
    batch_size = cfg["embedding"]["batch_size"]
    dim = cfg["embedding"]["pca_dim"]

    prep_dir = EXP_ROOT / "embeddings"
    prep_dir.mkdir(parents=True, exist_ok=True)
    ready_file = prep_dir / "ready.json"

    if ready_file.exists():
        print("Embedding preparation was already completed.")
    else:
        print("Device:", device)
        print("Encoding original text...")
        original_raw = embeddings.extract_text(
            text_versions["original_text"].tolist(),
            model_name,
            batch_size,
            device
        )

        print("\nEncoding masked text...")
        masked_raw = embeddings.extract_text(
            text_versions["modified_text"].tolist(),
            model_name,
            batch_size,
            device
        )

        assert original_raw.shape == masked_raw.shape
        assert np.isfinite(original_raw).all()
        assert np.isfinite(masked_raw).all()

        original_raw = embeddings.l2_normalize(original_raw)
        masked_raw = embeddings.l2_normalize(masked_raw)

        # Fit PCA ONCE on original text and transform both versions.
        print("\nApplying a shared PCA transformation...")
        if original_raw.shape[1] > dim:
            pca = PCA(n_components=dim, random_state=42)
            original_emb = pca.fit_transform(original_raw)
            masked_emb = pca.transform(masked_raw)
            joblib.dump(pca, prep_dir / "text_pca.joblib")
            variance = float(pca.explained_variance_ratio_.sum())
        else:
            original_emb = original_raw
            masked_emb = masked_raw
            variance = None

        original_emb = embeddings.l2_normalize(
            original_emb.astype(np.float32)
        )
        masked_emb = embeddings.l2_normalize(
            masked_emb.astype(np.float32)
        )

        # Each text version gets its own embedding directory.
        for version, matrix in [
            ("original", original_emb),
            ("masked", masked_emb)
        ]:
            folder = prep_dir / version
            folder.mkdir(parents=True, exist_ok=True)
            np.save(folder / "text_minilm.npy", matrix)

            # Copy identical image features and shuffle assignments.
            # Source files in the shared folder are only read.
            filenames = [
                "img_marqo.npy",
                "shuffle_type_seed42.npy",
                "shuffle_type_seed43.npy",
                "shuffle_type_seed44.npy"
            ]
            for filename in filenames:
                shutil.copy2(
                    source_paths.embeddings / filename,
                    folder / filename
                )

        info = {
            "model": model_name,
            "n_items": int(len(text_versions)),
            "raw_dim": int(original_raw.shape[1]),
            "output_dim": int(original_emb.shape[1]),
            "pca_fit": "original text only; shared by both versions",
            "explained_variance": variance,
            "device": device
        }

        with open(ready_file, "w", encoding="utf-8") as f:
            json.dump(info, f, indent=2)

        print("Embedding shape:", original_emb.shape)
        print("PCA explained variance:", variance)

    print("\nEmbedding preparation complete.")
    print("Saved to:", prep_dir)
else:
    print('Saved embeddings will be used.')


## 3. 학습
Opt-in only: four conditions × three seeds = 12 runs.

In [ ]:
if RUN_TRAINING:
    import copy
    import json
    import hashlib
    import subprocess
    from msrs_paper import two_tower

    assert ready_file.exists(), "Complete embedding preparation first."
    assert cfg["train"]["tuned"], "Team training settings are not finalized."

    experiment_cfg = copy.deepcopy(cfg)
    experiment_exps = copy.deepcopy(exps)

    # Verify that the selected conditions match the intended experiment.
    assert experiment_exps["conditions"]["C2"] == {
        "image": "marqo", "shuffle": None, "text": True
    }
    assert experiment_exps["conditions"]["C4"] == {
        "image": "marqo", "shuffle": "type", "text": True
    }

    # Record the exact settings. Stop if an attempted restart changes them.
    settings = {
        "cfg": experiment_cfg,
        "conditions": {
            name: experiment_exps["conditions"][name]
            for name in ["C2", "C4"]
        },
        "code_commit": subprocess.check_output(
            ["git", "-C", str(CORE_DIR), "rev-parse", "HEAD"],
            text=True
        ).strip(),
        "text_versions_sha256": hashlib.sha256(
            (EXP_ROOT / "text_versions.parquet").read_bytes()
        ).hexdigest()
    }

    settings_file = EXP_ROOT / "training_settings.json"
    if settings_file.exists():
        with open(settings_file, encoding="utf-8") as f:
            previous_settings = json.load(f)
        assert previous_settings == settings, (
            "Settings or code changed. Use a new experiment folder."
        )
    else:
        with open(settings_file, "w", encoding="utf-8") as f:
            json.dump(settings, f, ensure_ascii=False, indent=2)

    def experiment_paths(version):
        p = Paths(EXP_ROOT)

        # Read the team's existing data splits.
        p.raw = source_paths.raw
        p.processed = source_paths.processed

        # Read this experiment's text and image features.
        p.embeddings = EXP_ROOT / "embeddings" / version

        # Write predictions only inside this personal experiment.
        p.outputs.mkdir(parents=True, exist_ok=True)
        p.logs.mkdir(parents=True, exist_ok=True)
        return p

    for version in ["original", "masked"]:
        run_paths = experiment_paths(version)

        for seed in [42, 43, 44]:
            for condition in ["C2", "C4"]:
                print(
                    f"\n=== {version} | {condition} | seed {seed} ===",
                    flush=True
                )

                two_tower.run(
                    experiment_cfg,
                    experiment_exps,
                    run_paths,
                    cond=condition,
                    seed=seed,
                    split="main",
                    group=version,
                    device=device
                )

    print("\nAll 12 training runs completed.")
    print("Predictions saved to:", EXP_ROOT / "outputs")
else:
    print('Saved predictions will be used; training skipped.')


## 4. 재채점
Needs evaluation_products.csv, saved embeddings and all 12 prediction folders in Drive.

In [ ]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from msrs_paper import metrics

# EXP_ROOT is configured above.

SEEDS = [42, 43, 44]
K = 300
N_BOOT = 1000
BOOT_SEED = 42

# Load the evaluation set saved before training.
evaluation_products = pd.read_csv(
    EXP_ROOT / "evaluation_products.csv"
)
assert len(evaluation_products) == 5972
assert evaluation_products["idx"].is_unique

all_targets = pd.read_parquet(
    Path(DRIVE_ROOT) / "processed/main/test_targets.parquet"
)

selected = (
    all_targets["item_idx"].isin(evaluation_products["idx"])
    & all_targets["appearance"].eq("pattern")
    & ~all_targets["is_repurchase"]
)
evaluation_targets = all_targets.loc[selected].reset_index(drop=True)
assert len(evaluation_targets) > 0, "No evaluation purchases found."

# Use the same customer population and ordering for every condition.
all_users = np.sort(all_targets["user"].unique())
customer_rows = metrics.target_rows(all_users, evaluation_targets)
n_users = len(all_users)

purchases_per_user = np.bincount(
    customer_rows, minlength=n_users
)
n_purchases = int(purchases_per_user.sum())

print("Evaluated products with test purchases:",
      evaluation_targets["item_idx"].nunique())
print("Evaluated non-repurchase purchases:", n_purchases)
print("Customers with evaluated purchases:",
      int((purchases_per_user > 0).sum()))

print("\nPurchase counts by category:")
print(evaluation_targets["pgroup"].value_counts().to_string())

# Check that masking actually changed the stored text embeddings.
original_emb = np.load(
    EXP_ROOT / "embeddings/original/text_minilm.npy",
    mmap_mode="r"
)
masked_emb = np.load(
    EXP_ROOT / "embeddings/masked/text_minilm.npy",
    mmap_mode="r"
)
assert original_emb.shape == masked_emb.shape
assert np.isfinite(original_emb).all()
assert np.isfinite(masked_emb).all()

selected_indices = evaluation_products["idx"].to_numpy(dtype=int)
mean_change = float(np.mean(np.linalg.norm(
    original_emb[selected_indices] - masked_emb[selected_indices],
    axis=1
)))
assert mean_change > 0, "Original and masked embeddings are identical."
print("\nMean embedding change for selected products:", mean_change)

def load_hits(version, condition, seed):
    folder = (
        EXP_ROOT / "outputs" / version / "main"
        / f"{condition}_seed{seed}"
    )

    with open(folder / "run_info.json", encoding="utf-8") as f:
        info = json.load(f)

    assert info["seed"] == seed
    assert info["cond"] == condition
    assert info["split"] == "main"
    assert info["group"] == version

    users = np.load(folder / "users.npy", allow_pickle=False)
    candidates = np.load(
        folder / "candidates.npy",
        mmap_mode="r",
        allow_pickle=False
    )

    assert users.ndim == 1
    assert len(np.unique(users)) == len(users)
    assert candidates.ndim == 2
    assert candidates.shape[0] == len(users)
    assert candidates.shape[1] >= K

    rows = metrics.target_rows(users, evaluation_targets)
    hits = metrics.hits_at_k(
        candidates,
        rows,
        evaluation_targets["item_idx"].to_numpy(),
        K
    ).astype(np.int64)

    customer_hits = np.bincount(
        customer_rows,
        weights=hits,
        minlength=n_users
    )
    return customer_hits, info

# Store customer hit counts for all four conditions and three seeds.
hits_by_condition = {
    (version, condition): []
    for version in ["original", "masked"]
    for condition in ["C2", "C4"]
}

reference_params = None
reference_pairs = None

for seed in SEEDS:
    for version in ["original", "masked"]:
        for condition in ["C2", "C4"]:
            print(f"Scoring {version}, {condition}, seed {seed}...")
            hits, info = load_hits(version, condition, seed)

            if reference_params is None:
                reference_params = info["params"]
                reference_pairs = info["n_train_pairs"]

            assert info["params"] == reference_params, (
                "Training settings differ between runs."
            )
            assert info["n_train_pairs"] == reference_pairs, (
                "Training sample counts differ between runs."
            )

            hits_by_condition[(version, condition)].append(hits)

mean_hits = {
    key: np.mean(np.stack(values), axis=0)
    for key, values in hits_by_condition.items()
}

scores = {
    key: float(values.sum() / n_purchases)
    for key, values in mean_hits.items()
}

original_contribution = (
    scores[("original", "C2")] - scores[("original", "C4")]
)
masked_contribution = (
    scores[("masked", "C2")] - scores[("masked", "C4")]
)
estimate = masked_contribution - original_contribution

# Customer-level difference of differences.
customer_contrast = (
    mean_hits[("masked", "C2")]
    - mean_hits[("masked", "C4")]
    - mean_hits[("original", "C2")]
    + mean_hits[("original", "C4")]
)

# Use the same sampled customers for all four conditions.
rng = np.random.default_rng(BOOT_SEED)
bootstrap_estimates = []

for _ in range(N_BOOT):
    sampled = rng.integers(0, n_users, size=n_users)
    multiplicity = np.bincount(sampled, minlength=n_users)
    denominator = multiplicity @ purchases_per_user

    if denominator > 0:
        bootstrap_estimates.append(
            float((multiplicity @ customer_contrast) / denominator)
        )

assert len(bootstrap_estimates) >= 950
low, high = np.quantile(bootstrap_estimates, [0.025, 0.975])

# Show how consistent the difference is across training seeds.
seed_rows = []
for i, seed in enumerate(SEEDS):
    seed_scores = {
        key: float(values[i].sum() / n_purchases)
        for key, values in hits_by_condition.items()
    }
    original_gain = (
        seed_scores[("original", "C2")]
        - seed_scores[("original", "C4")]
    )
    masked_gain = (
        seed_scores[("masked", "C2")]
        - seed_scores[("masked", "C4")]
    )

    seed_rows.append({
        "seed": seed,
        "original_contribution": original_gain,
        "masked_contribution": masked_gain,
        "difference": masked_gain - original_gain
    })

score_table = pd.DataFrame([
    {
        "text_version": version,
        "real_recall_at_300": scores[(version, "C2")],
        "shuffled_recall_at_300": scores[(version, "C4")],
        "image_contribution": (
            scores[(version, "C2")] - scores[(version, "C4")]
        )
    }
    for version in ["original", "masked"]
])
seed_table = pd.DataFrame(seed_rows)

summary = {
    "hypothesis_id": "H_TextMask",
    "status": "exploratory",
    "measure": "Purchase-level Recall@300",
    "segment": "Non-repurchase purchases of fixed changed pattern products",
    "n_fixed_products": int(len(evaluation_products)),
    "n_products_with_test_purchases": int(
        evaluation_targets["item_idx"].nunique()
    ),
    "n_test_purchases": n_purchases,
    "n_evaluated_customers": int((purchases_per_user > 0).sum()),
    "original_image_contribution": original_contribution,
    "masked_image_contribution": masked_contribution,
    "difference_masked_minus_original": estimate,
    "ci95_customer_bootstrap": [float(low), float(high)],
    "training_seeds": SEEDS,
    "bootstrap_samples": len(bootstrap_estimates),
    "bootstrap_seed": BOOT_SEED,
    "interpretation_limit": (
        "Customer bootstrap conditional on the trained models;"
        " effects are specific to this keyword-masking procedure."
    )
}

# Create a fresh results folder for this evaluation.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
result_dir = RUN_OUTPUT_ROOT / "results"
result_dir.mkdir(parents=True, exist_ok=False)

score_table.to_csv(result_dir / "condition_scores.csv", index=False)
seed_table.to_csv(result_dir / "seed_scores.csv", index=False)
np.save(result_dir / "bootstrap_estimates.npy", bootstrap_estimates)

with open(result_dir / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nCondition results:")
print(score_table.round(6).to_string(index=False))

print("\nPer-seed results:")
print(seed_table.round(6).to_string(index=False))

print(f"\nMasked minus original contribution: {estimate:+.6f}")
print(f"95% customer-bootstrap interval: [{low:+.6f}, {high:+.6f}]")
print("\nSaved to:", result_dir)